In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/AKI Project"

os.listdir(DATA_PATH)

['patient.csv',
 'procedure.csv',
 'diagnosis.csv',
 'encounter.csv',
 'lab_result.csv',
 'patient_cohort.csv',
 'tumor_properties.csv',
 'medication_drug.csv',
 'oncology_treatment.csv',
 'genomic.csv',
 'chemo_lines.csv',
 'dataset_details.csv',
 'medication_ingredient.csv',
 'tumor.csv',
 'vitals_signs.csv',
 'cohort_details.csv',
 'manifest.csv',
 'standardized_terminology.csv',
 'AKI_Project.ipynb']

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
diag_path = os.path.join(DATA_PATH, "diagnosis.csv")

diagnosis = pd.read_csv(diag_path)

diagnosis.head()

,patient_id,encounter_id,code_system,code,principal_diagnosis_indicator,admitting_diagnosis,reason_for_visit,date,derived_by_TriNetX,source_id
0,fQ,iQB,ICD-10-CM,B07.8,P,U,U,20190701,F,EHR
1,fQ,iQN,ICD-10-CM,B07.8,P,U,U,20190708,F,EHR
2,fQ,iQB,ICD-10-CM,B36.0,S,U,U,20190701,F,EHR
3,fQ,iga,ICD-10-CM,B44.9,S,U,U,20231026,F,EHR
4,fQ,iwY,ICD-10-CM,B44.9,P,U,U,20231102,F,EHR


In [ ]:
diagnosis.shape
diagnosis['code_system'].value_counts()

,count
code_system,
ICD-10-CM,47978634
ICD-9-CM,16448262


In [ ]:
# AKI ICD-10 (N17*)
aki_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'].str.startswith('N17', na=False))
]

# AKI ICD-9 (584*)
aki_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'].str.startswith('584', na=False))
]

print("AKI ICD-10 count:", aki_icd10.shape[0])
print("AKI ICD-9 count:", aki_icd9.shape[0])

AKI ICD-10 count: 248770
AKI ICD-9 count: 57538


In [ ]:
# Combine ICD-10 and ICD-9 AKI rows
aki_all = pd.concat([aki_icd10, aki_icd9])

print("Total AKI diagnosis rows:", aki_all.shape[0])

Total AKI diagnosis rows: 306308


In [ ]:
aki_all['date'] = pd.to_datetime(aki_all['date'], format='%Y%m%d', errors='coerce')

aki_all.head()

,patient_id,encounter_id,code_system,code,principal_diagnosis_indicator,admitting_diagnosis,reason_for_visit,date,derived_by_TriNetX,source_id
1651,HRC,KhVC,ICD-10-CM,N17.9,Unknown,U,U,2009-02-14,F,EHR
2425,HBD,KBgD,ICD-10-CM,N17.9,Unknown,U,U,2010-03-27,F,EHR
2426,HBD,KhfD,ICD-10-CM,N17.9,Unknown,U,U,2010-03-28,F,EHR
2910,HhD,KxxD,ICD-10-CM,N17.9,Unknown,U,U,2011-04-11,F,EHR
3140,HRE,KRUE,ICD-10-CM,N17.9,Unknown,U,U,2021-01-08,F,EHR


In [ ]:
aki_index = (
    aki_all
    .sort_values('date')
    .groupby('patient_id')
    .first()
    .reset_index()[['patient_id', 'date']]
)

aki_index.rename(columns={'date': 'aki_index_date'}, inplace=True)

print("Number of unique AKI patients:", aki_index.shape[0])
aki_index.head()

Number of unique AKI patients: 33323


,patient_id,aki_index_date
0,#A#B,2016-05-23
1,#A#C,2021-10-16
2,#A#D,2022-07-01
3,#A0B,2018-04-16
4,#A0C,2012-10-18


# EXCLUDING ESRD PATIENTS DIAGNOSED BEFORE AKI

Patients diagnosed with ESRD before or on the AKI index date are excluded because:

- ESRD represents the outcome (end-stage kidney failure).
- The study aims to predict **future** progression to ESRD after AKI.
- Including prior ESRD would violate temporal order.
- It would introduce data leakage and artificially inflate model performance.

This ensures a clean and clinically valid AKI progression cohort.

In [ ]:
# ESRD ICD-10 (exact match)
esrd_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'] == 'N18.6')
]

# ESRD ICD-9 (exact match)
esrd_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'] == '585.6')
]

esrd_all = pd.concat([esrd_icd10, esrd_icd9])

print("Total ESRD diagnosis rows:", esrd_all.shape[0])

Total ESRD diagnosis rows: 308031


In [ ]:
# Ensure AKI index date is datetime
aki_index['aki_index_date'] = pd.to_datetime(aki_index['aki_index_date'], errors='coerce')

# Ensure ESRD date is datetime
esrd_all['date'] = pd.to_datetime(esrd_all['date'], format='%Y%m%d', errors='coerce')

In [ ]:
esrd_in_aki = esrd_all.merge(aki_index, on='patient_id', how='inner')
print("ESRD rows among AKI patients:", esrd_in_aki.shape[0])
print("AKI patients with ANY ESRD at any time:", esrd_in_aki['patient_id'].nunique())

ESRD rows among AKI patients: 197254
AKI patients with ANY ESRD at any time: 4725


In [ ]:
pre_existing_esrd = esrd_in_aki[esrd_in_aki['date'] <= esrd_in_aki['aki_index_date']]

print("Pre-existing ESRD rows (<= index):", pre_existing_esrd.shape[0])
print("Patients with pre-existing ESRD:", pre_existing_esrd['patient_id'].nunique())

Pre-existing ESRD rows (<= index): 34445
Patients with pre-existing ESRD: 1924


In [ ]:
exclude_patients = set(pre_existing_esrd['patient_id'].unique())

aki_clean = aki_index[~aki_index['patient_id'].isin(exclude_patients)].copy()

print("AKI patients before exclusion:", aki_index.shape[0])
print("AKI patients after ESRD exclusion:", aki_clean.shape[0])

AKI patients before exclusion: 33323
AKI patients after ESRD exclusion: 31399


In [ ]:
print("Any excluded patients still in cohort?",
      aki_clean['patient_id'].isin(exclude_patients).any())

Any excluded patients still in cohort? False


# Defining 2 year prediction window

### index date and end date for each patient

In [ ]:
aki_clean['prediction_end_date'] = aki_clean['aki_index_date'] + pd.DateOffset(years=2)

aki_clean.head()

,patient_id,aki_index_date,prediction_end_date
0,#A#B,2016-05-23,2018-05-23
1,#A#C,2021-10-16,2023-10-16
2,#A#D,2022-07-01,2024-07-01
3,#A0B,2018-04-16,2020-04-16
4,#A0C,2012-10-18,2014-10-18


In [ ]:
aki_clean.shape[0]

31399

In [ ]:
# Merge ESRD with clean AKI cohort
esrd_future = esrd_all.merge(aki_clean, on='patient_id', how='inner')

# Keep ESRD that happens AFTER index and WITHIN 2 years
esrd_future = esrd_future[
    (esrd_future['date'] > esrd_future['aki_index_date']) &
    (esrd_future['date'] <= esrd_future['prediction_end_date'])
]

print("Patients who progressed to ESRD within 2 years:",
      esrd_future['patient_id'].nunique())

Patients who progressed to ESRD within 2 years: 1721


In [ ]:
# creating a supervised learning target
# creating outcome label: 1 if progressed to ESRD within 2 years, else 0
esrd_patients_2yr = set(esrd_future['patient_id'].unique())

aki_clean['Y_esrd_2yr'] = aki_clean['patient_id'].isin(esrd_patients_2yr).astype(int)

print(aki_clean['Y_esrd_2yr'].value_counts())
print("Positive rate:", aki_clean['Y_esrd_2yr'].mean())

Y_esrd_2yr
0    29678
1     1721
Name: count, dtype: int64
Positive rate: 0.05481066275996051


Loading patient.csv to inspect code systems

In [ ]:
proc_path = os.path.join(DATA_PATH, "procedure.csv")

# Use dtype=str so codes like '585.6' or codes with leading zeros don't get mangled
procedure = pd.read_csv(proc_path, dtype=str)

print(procedure.shape)
print(procedure['code_system'].value_counts())

procedure.head()

(44334752, 8)
code_system
CPT           35371873
HCPCS          6120985
SNOMED         1776498
ICD-10-PCS      570581
ICD-9-CM        494815
Name: count, dtype: int64


,patient_id,encounter_id,code_system,code,principal_procedure_indicator,date,derived_by_TriNetX,source_id
0,fQ,iQ,CPT,11102,Unknown,20220831,F,EHR
1,fQ,ig,CPT,11102,Unknown,20230919,F,EHR
2,fQ,iQ,CPT,11103,Unknown,20220831,F,EHR
3,fQ,iw,CPT,11200,Unknown,20151007,F,EHR
4,fQ,iAB,CPT,15852,Unknown,20221130,F,EHR


In [ ]:
# =========================
# DIALYSIS outcome within 2 years
# =========================

# 0) Ensure datetime columns exist
aki_clean = aki_clean.copy()
aki_clean["aki_index_date"] = pd.to_datetime(aki_clean["aki_index_date"], errors="coerce")
if "prediction_end_date" not in aki_clean.columns:
    aki_clean["prediction_end_date"] = aki_clean["aki_index_date"] + pd.DateOffset(years=2)

procedure = procedure.copy()
procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")

# 1) Dialysis code definitions (common starters; you can expand later)
CPT_DIALYSIS = {"90935", "90937", "90945", "90947"}      # dialysis procedures
HCPCS_DIALYSIS = {"G0257"}                               # unscheduled dialysis (common)
ICD10PCS_PREFIXES = ("5A1D",)                            # hemodialysis (ICD-10-PCS)
ICD9PROC_DIALYSIS = {"39.95"}                            # hemodialysis (ICD-9 procedure)

# normalize code_system to avoid case issues
procedure["code_system_norm"] = procedure["code_system"].astype(str).str.upper().str.strip()
procedure["code_norm"] = procedure["code"].astype(str).str.strip()

In [ ]:
# 2) Pull dialysis rows from procedure table
dialysis_cpt = procedure[
    (procedure["code_system_norm"].str.contains("CPT", na=False)) &
    (procedure["code_norm"].isin(CPT_DIALYSIS))
]

dialysis_hcpcs = procedure[
    (procedure["code_system_norm"].str.contains("HCPCS", na=False)) &
    (procedure["code_norm"].isin(HCPCS_DIALYSIS))
]

dialysis_icd10pcs = procedure[
    (procedure["code_system_norm"].str.contains("ICD-10", na=False)) &
    (procedure["code_system_norm"].str.contains("PCS", na=False)) &
    (procedure["code_norm"].str.startswith(ICD10PCS_PREFIXES, na=False))
]

dialysis_icd9proc = procedure[
    (procedure["code_system_norm"].str.contains("ICD-9", na=False)) &
    (procedure["code_norm"].isin(ICD9PROC_DIALYSIS))
]

dialysis_all = pd.concat(
    [dialysis_cpt, dialysis_hcpcs, dialysis_icd10pcs, dialysis_icd9proc],
    ignore_index=True
)

print("Total dialysis procedure rows (all patients):", dialysis_all.shape[0])

Total dialysis procedure rows (all patients): 100002


In [ ]:
# 3) Restrict to AKI cohort and 2-year window
dialysis_in_aki = dialysis_all.merge(
    aki_clean[["patient_id", "aki_index_date", "prediction_end_date"]],
    on="patient_id",
    how="inner"
)

dialysis_future = dialysis_in_aki[
    (dialysis_in_aki["date"] >= dialysis_in_aki["aki_index_date"]) &
    (dialysis_in_aki["date"] <= dialysis_in_aki["prediction_end_date"])
].copy()

dialysis_patients_2yr = set(dialysis_future["patient_id"].unique())
print("AKI patients with ANY dialysis within 2 years (incl acute):", len(dialysis_patients_2yr))

AKI patients with ANY dialysis within 2 years (incl acute): 2448


In [ ]:
# 4) Create dialysis label
aki_clean["Y_dialysis_2yr"] = aki_clean["patient_id"].isin(dialysis_patients_2yr).astype(int)
print(aki_clean["Y_dialysis_2yr"].value_counts())
print("Dialysis positive rate:", aki_clean["Y_dialysis_2yr"].mean())

# 5) Combine ESRD OR dialysis label (what Kalyani did)
# (Assumes you already created Y_esrd_2yr; recreate esrd_patients_2yr if you need)
esrd_patients_2yr = set(aki_clean.loc[aki_clean["Y_esrd_2yr"] == 1, "patient_id"])
combined_patients_2yr = esrd_patients_2yr | dialysis_patients_2yr

aki_clean["Y_esrd_or_dialysis_2yr"] = aki_clean["patient_id"].isin(combined_patients_2yr).astype(int)
print(aki_clean["Y_esrd_or_dialysis_2yr"].value_counts())
print("Combined positive rate:", aki_clean["Y_esrd_or_dialysis_2yr"].mean())

Y_dialysis_2yr
0    28951
1     2448
Name: count, dtype: int64
Dialysis positive rate: 0.07796426637791012
Y_esrd_or_dialysis_2yr
0    28208
1     3191
Name: count, dtype: int64
Combined positive rate: 0.10162744036434282


# Building the observation window

In [ ]:
aki_clean["obs_start_date"] = aki_clean["aki_index_date"] - pd.DateOffset(years=1)

In [ ]:
aki_clean.head()

,patient_id,aki_index_date,prediction_end_date,Y_esrd_2yr,Y_dialysis_2yr,Y_esrd_or_dialysis_2yr,obs_start_date
0,#A#B,2016-05-23,2018-05-23,0,0,0,2015-05-23
1,#A#C,2021-10-16,2023-10-16,1,1,1,2020-10-16
2,#A#D,2022-07-01,2024-07-01,0,0,0,2021-07-01
3,#A0B,2018-04-16,2020-04-16,0,0,0,2017-04-16
4,#A0C,2012-10-18,2014-10-18,0,0,0,2011-10-18


In [ ]:
aki_clean.shape

(31399, 7)

In [ ]:
# filter diagnosis in the window
diagnosis["date"] = pd.to_datetime(diagnosis["date"], format="%Y%m%d", errors="coerce")

diag_obs = diagnosis.merge(
    aki_clean[["patient_id","obs_start_date","aki_index_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    (diag_obs["date"] >= diag_obs["obs_start_date"]) &
    (diag_obs["date"] < diag_obs["aki_index_date"])
]

In [ ]:
diag_obs.head(20)

,patient_id,encounter_id,code_system,code,principal_diagnosis_indicator,admitting_diagnosis,reason_for_visit,date,derived_by_TriNetX,source_id,obs_start_date,aki_index_date
87,HBD,KRfD,ICD-10-CM,R06.9,Unknown,U,U,2010-03-13,F,EHR,2009-03-27,2010-03-27
99,HBD,KBeD,ICD-10-CM,R52,Unknown,U,U,2010-03-18,F,EHR,2009-03-27,2010-03-27
100,HBD,KhfD,ICD-10-CM,R52,Unknown,U,U,2010-03-20,F,EHR,2009-03-27,2010-03-27
103,HBD,KBeD,ICD-10-CM,J18.9,Unknown,U,U,2010-03-12,F,EHR,2009-03-27,2010-03-27
104,HBD,KBeD,ICD-10-CM,J18.9,Unknown,U,U,2010-03-13,F,EHR,2009-03-27,2010-03-27
109,HBD,KhfD,ICD-10-CM,J18.1,Unknown,U,U,2010-03-25,F,EHR,2009-03-27,2010-03-27
110,HBD,KBeD,ICD-10-CM,R06.89,Unknown,U,U,2010-03-12,F,EHR,2009-03-27,2010-03-27
111,HBD,KhfD,ICD-10-CM,I31.9,Unknown,U,U,2010-03-25,F,EHR,2009-03-27,2010-03-27
114,HBD,KhfD,ICD-10-CM,I31.4,Unknown,U,U,2010-03-21,F,EHR,2009-03-27,2010-03-27
125,HBD,KBeD,ICD-10-CM,D57.01,Unknown,U,U,2010-03-12,F,EHR,2009-03-27,2010-03-27


In [ ]:
diag_obs["code"].nunique()

22617

# Tabular feature engineering (simpler baseline approach)
- To make sure the cohort is correct
- to check if signals/indicators exist
- gives a benchmark model

Later I will progress to sequences like Kalyani did so I could compare the improvement over this simple baseline

In [ ]:
# finding the 300 top most frequent diagnoses codes and arranging them in a table with the codes as columns (300 columns)

top_codes = (
    diag_obs["code"]
    .value_counts()
    .head(300)
    .index
)

In [ ]:
diag_obs_top = diag_obs[diag_obs["code"].isin(top_codes)]

In [ ]:
diag_features = (
    diag_obs_top
    .groupby(["patient_id", "code"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
diag_features.head()

code,162.9,174.9,185,203.00,205.00,238.75,244.9,250.00,250.02,250.60,272.0,272.2,272.4,278.00,278.01,285.9,300.00,305.1,311,327.23,338.29,401.1,401.9,414.00,414.01,424.1,425.4,427.31,428.0,443.9,486,493.90,496,530.81,571.5,585.3,585.9,593.9,599.0,724.2,729.5,780.79,782.3,786.05,786.09,786.2,786.50,787.91,789.00,A41.9,B18.2,B20,B35.1,C15.9,C18.9,C20,C22.0,C22.1,C25.9,C34.90,C50.9,C54.1,C56.9,C61,C64.9,C67.9,C78.00,C78.7,C79.31,C79.51,C80.1,C83.30,C90.00,C91.00,C91.10,C92.00,D46.9,D50.0,D50.9,D61.818,D62,D63.1,D64.9,D69.6,D70.9,D72.829,D84.9,E03.9,E11.22,E11.40,E11.42,E11.621,E11.65,E11.69,E11.8,E11.9,E53.8,E55.9,E66.01,E66.9,E78.00,E78.2,E78.5,E83.42,E86.0,E87.1,E87.5,E87.6,F03.90,F10.10,F17.200,F17.210,F31.9,F32.9,F32.A,F41.1,F41.9,G40.909,G47.00,G47.30,G47.33,G62.9,G89.29,G89.3,G89.4,I10,I11.0,I12.9,I13.0,I21.4,I25.10,I25.2,I25.5,I26.99,I27.20,I34.0,I35.0,I42.8,I42.9,I48.0,I48.19,I48.2,I48.20,I48.91,I48.92,I50.20,I50.22,I50.23,I50.30,I50.32,I50.33,I50.42,I50.9,I51.7,I63.9,I73.9,I87.2,I89.0,I95.9,J18.9,J30.9,J43.9,J44.1,J44.9,J45.909,J84.9,J90,J96.01,J98.11,J98.4,K21.9,K44.9,K52.9,K57.30,K59.00,K70.31,K74.60,K76.0,K92.2,M06.9,M10.9,M17.0,M19.90,M25.511,M25.551,M25.561,M25.562,M47.816,M48.061,M51.36,M54.16,M54.2,M54.5,M54.50,M54.9,M79.89,M81.0,M85.80,N18.3,N18.30,N18.4,N18.9,N20.0,N28.89,N28.9,N39.0,N40.0,N40.1,R00.0,R00.1,R05,R05.9,R06.00,R06.02,R06.09,R07.89,R07.9,R09.02,R09.89,R10.13,R10.9,R11.0,R11.2,R13.10,R18.8,R19.7,R21,R26.2,R29.898,R30.0,R31.9,R33.9,R41.82,R42,R50.9,R51,R51.9,R52,R53.1,R53.81,R53.83,R55,R56.9,R59.0,R60.0,R60.9,R63.4,R73.03,R73.9,R78.81,R79.89,R80.9,R91.1,R91.8,R94.31,T45.1X5A,U07.1,V45.89,V58.11,V58.61,V58.69,W19.XXXA,Z00.00,Z01.818,Z09,Z12.11,Z12.31,Z20.822,Z23,Z45.2,Z51.0,Z51.11,Z51.12,Z51.81,Z68.41,Z72.0,Z79.01,Z79.02,Z79.4,Z79.52,Z79.82,Z79.84,Z79.899,Z86.711,Z86.718,Z86.73,Z87.891,Z88.0,Z90.49,Z92.21,Z92.3,Z94.2,Z94.81,Z95.0,Z95.1,Z95.2,Z95.5,Z95.810,Z98.890,Z99.81
patient_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
#A#B,0,0,0,0,0,0,0,1,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,3,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
#A#C,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,1,4,0,1,0,0,0,4,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,4,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
#A#D,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,3,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,

In [ ]:
# merging with the AKI cohort
X = aki_clean[["patient_id"]].merge(
    diag_features,
    on="patient_id",
    how="left"
)

X = X.fillna(0)

In [ ]:
X.head()

,patient_id,162.9,174.9,185,203.00,205.00,238.75,244.9,250.00,250.02,250.60,272.0,272.2,272.4,278.00,278.01,285.9,300.00,305.1,311,327.23,338.29,401.1,401.9,414.00,414.01,424.1,425.4,427.31,428.0,443.9,486,493.90,496,530.81,571.5,585.3,585.9,593.9,599.0,724.2,729.5,780.79,782.3,786.05,786.09,786.2,786.50,787.91,789.00,A41.9,B18.2,B20,B35.1,C15.9,C18.9,C20,C22.0,C22.1,C25.9,C34.90,C50.9,C54.1,C56.9,C61,C64.9,C67.9,C78.00,C78.7,C79.31,C79.51,C80.1,C83.30,C90.00,C91.00,C91.10,C92.00,D46.9,D50.0,D50.9,D61.818,D62,D63.1,D64.9,D69.6,D70.9,D72.829,D84.9,E03.9,E11.22,E11.40,E11.42,E11.621,E11.65,E11.69,E11.8,E11.9,E53.8,E55.9,E66.01,E66.9,E78.00,E78.2,E78.5,E83.42,E86.0,E87.1,E87.5,E87.6,F03.90,F10.10,F17.200,F17.210,F31.9,F32.9,F32.A,F41.1,F41.9,G40.909,G47.00,G47.30,G47.33,G62.9,G89.29,G89.3,G89.4,I10,I11.0,I12.9,I13.0,I21.4,I25.10,I25.2,I25.5,I26.99,I27.20,I34.0,I35.0,I42.8,I42.9,I48.0,I48.19,I48.2,I48.20,I48.91,I48.92,I50.20,I50.22,I50.23,I50.30,I50.32,I50.33,I50.42,I50.9,I51.7,I63.9,I73.9,I87.2,I89.0,I95.9,J18.9,J30.9,J43.9,J44.1,J44.9,J45.909,J84.9,J90,J96.01,J98.11,J98.4,K21.9,K44.9,K52.9,K57.30,K59.00,K70.31,K74.60,K76.0,K92.2,M06.9,M10.9,M17.0,M19.90,M25.511,M25.551,M25.561,M25.562,M47.816,M48.061,M51.36,M54.16,M54.2,M54.5,M54.50,M54.9,M79.89,M81.0,M85.80,N18.3,N18.30,N18.4,N18.9,N20.0,N28.89,N28.9,N39.0,N40.0,N40.1,R00.0,R00.1,R05,R05.9,R06.00,R06.02,R06.09,R07.89,R07.9,R09.02,R09.89,R10.13,R10.9,R11.0,R11.2,R13.10,R18.8,R19.7,R21,R26.2,R29.898,R30.0,R31.9,R33.9,R41.82,R42,R50.9,R51,R51.9,R52,R53.1,R53.81,R53.83,R55,R56.9,R59.0,R60.0,R60.9,R63.4,R73.03,R73.9,R78.81,R79.89,R80.9,R91.1,R91.8,R94.31,T45.1X5A,U07.1,V45.89,V58.11,V58.61,V58.69,W19.XXXA,Z00.00,Z01.818,Z09,Z12.11,Z12.31,Z20.822,Z23,Z45.2,Z51.0,Z51.11,Z51.12,Z51.81,Z68.41,Z72.0,Z79.01,Z79.02,Z79.4,Z79.52,Z79.82,Z79.84,Z79.899,Z86.711,Z86.718,Z86.73,Z87.891,Z88.0,Z90.49,Z92.21,Z92.3,Z94.2,Z94.81,Z95.0,Z95.1,Z95.2,Z95.5,Z95.810,Z98.890,Z99.81
0,#A#B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,#A#C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,4.0,0.0,1.0,0.0,0.0,0.0,4.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0

In [ ]:
# adding the target (1 if the patient developed esrd or needed dialysis within 2 years of AKI diagnoses)
# this becomes the main modeling dataset where columns= diseases 1 year before AKI and target= esrd/dialysis within 2 years
# i will run classification on this dataset
X["target"] = aki_clean["Y_esrd_or_dialysis_2yr"]

In [ ]:
X.head()

,patient_id,162.9,174.9,185,203.00,205.00,238.75,244.9,250.00,250.02,250.60,272.0,272.2,272.4,278.00,278.01,285.9,300.00,305.1,311,327.23,338.29,401.1,401.9,414.00,414.01,424.1,425.4,427.31,428.0,443.9,486,493.90,496,530.81,571.5,585.3,585.9,593.9,599.0,724.2,729.5,780.79,782.3,786.05,786.09,786.2,786.50,787.91,789.00,A41.9,B18.2,B20,B35.1,C15.9,C18.9,C20,C22.0,C22.1,C25.9,C34.90,C50.9,C54.1,C56.9,C61,C64.9,C67.9,C78.00,C78.7,C79.31,C79.51,C80.1,C83.30,C90.00,C91.00,C91.10,C92.00,D46.9,D50.0,D50.9,D61.818,D62,D63.1,D64.9,D69.6,D70.9,D72.829,D84.9,E03.9,E11.22,E11.40,E11.42,E11.621,E11.65,E11.69,E11.8,E11.9,E53.8,E55.9,E66.01,E66.9,E78.00,E78.2,E78.5,E83.42,E86.0,E87.1,E87.5,E87.6,F03.90,F10.10,F17.200,F17.210,F31.9,F32.9,F32.A,F41.1,F41.9,G40.909,G47.00,G47.30,G47.33,G62.9,G89.29,G89.3,G89.4,I10,I11.0,I12.9,I13.0,I21.4,I25.10,I25.2,I25.5,I26.99,I27.20,I34.0,I35.0,I42.8,I42.9,I48.0,I48.19,I48.2,I48.20,I48.91,I48.92,I50.20,I50.22,I50.23,I50.30,I50.32,I50.33,I50.42,I50.9,I51.7,I63.9,I73.9,I87.2,I89.0,I95.9,J18.9,J30.9,J43.9,J44.1,J44.9,J45.909,J84.9,J90,J96.01,J98.11,J98.4,K21.9,K44.9,K52.9,K57.30,K59.00,K70.31,K74.60,K76.0,K92.2,M06.9,M10.9,M17.0,M19.90,M25.511,M25.551,M25.561,M25.562,M47.816,M48.061,M51.36,M54.16,M54.2,M54.5,M54.50,M54.9,M79.89,M81.0,M85.80,N18.3,N18.30,N18.4,N18.9,N20.0,N28.89,N28.9,N39.0,N40.0,N40.1,R00.0,R00.1,R05,R05.9,R06.00,R06.02,R06.09,R07.89,R07.9,R09.02,R09.89,R10.13,R10.9,R11.0,R11.2,R13.10,R18.8,R19.7,R21,R26.2,R29.898,R30.0,R31.9,R33.9,R41.82,R42,R50.9,R51,R51.9,R52,R53.1,R53.81,R53.83,R55,R56.9,R59.0,R60.0,R60.9,R63.4,R73.03,R73.9,R78.81,R79.89,R80.9,R91.1,R91.8,R94.31,T45.1X5A,U07.1,V45.89,V58.11,V58.61,V58.69,W19.XXXA,Z00.00,Z01.818,Z09,Z12.11,Z12.31,Z20.822,Z23,Z45.2,Z51.0,Z51.11,Z51.12,Z51.81,Z68.41,Z72.0,Z79.01,Z79.02,Z79.4,Z79.52,Z79.82,Z79.84,Z79.899,Z86.711,Z86.718,Z86.73,Z87.891,Z88.0,Z90.49,Z92.21,Z92.3,Z94.2,Z94.81,Z95.0,Z95.1,Z95.2,Z95.5,Z95.810,Z98.890,Z99.81,target
0,#A#B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,#A#C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,4.0,0.0,1.0,0.0,0.0,0.0,4.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

# ML Modeling

In [ ]:
# separating features and target
# features
X_model = X.drop(columns=["patient_id", "target"])

# target
y_model = X["target"]

print(X_model.shape)
print(y_model.shape)

(31399, 300)
(31399,)


In [ ]:
y_model.isna().sum()

np.int64(1785)

In [ ]:
# removing nan from target variable
mask = y_model.notna()

X_model = X_model[mask]
y_model = y_model[mask]

print("New shape:", X_model.shape)

New shape: (29614, 300)


In [ ]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42,
    stratify=y_model
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (23691, 300)
Test shape: (5923, 300)


In [ ]:
y_model.value_counts()

,count
target,
0.0,26626
1.0,2988


In [ ]:
# starting with LR as it is the standard baseline for classification
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.8968428161404693
ROC AUC: 0.49699389200307753
F1 Score: 0.0032626427406199023
              precision    recall  f1-score   support

         0.0       0.90      1.00      0.95      5325
         1.0       0.07      0.00      0.00       598

    accuracy                           0.90      5923
   macro avg       0.48      0.50      0.47      5923
weighted avg       0.81      0.90      0.85      5923



### above results show the dataset is heavily imbalanced and the accuracy is misleading.
- ROC AUC being 0.5 means the model is randomly guessing.
- F1 Score: 0.003 (extremely low) because recall=0.0 which means the model almost never predicts dialysis patients

# CLASS IMBALANCE FIX
- applying class weights to fix the class imbalance issue

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.4173560695593449
ROC AUC: 0.4943585033052271
F1 Score: 0.17301701413850948
              precision    recall  f1-score   support

         0.0       0.90      0.40      0.55      5325
         1.0       0.10      0.60      0.17       598

    accuracy                           0.42      5923
   macro avg       0.50      0.50      0.36      5923
weighted avg       0.82      0.42      0.51      5923



- this is a big improvement as recall 0.6 which means the model is finally detecting dialysis patients (model finds 60% of real dialysis patients)
- precision = 0.1 (only 10% of predicted dialysis are correct)
- F1 = 0.17 overall balance
- ROC AUC is still bad as the model cannot separate low-risk vs high-risk patients

# NEXT improvements
- adding procedure features
- adding medication features
- adding CKD stage features
- using better models

# EXPLORING THE LAB RESULT CSV

In [ ]:
lab = pd.read_csv(DATA_PATH + "/lab_result.csv", nrows=1)

print(lab.columns)

Index(['patient_id', 'encounter_id', 'code_system', 'code', 'date',
       'lab_result_num_val', 'lab_result_text_val', 'units_of_measure',
       'derived_by_TriNetX', 'source_id'],
      dtype='object')


In [ ]:
lab_sample = pd.read_csv(DATA_PATH + "/lab_result.csv", nrows=1000)

lab_sample.head()

,patient_id,encounter_id,code_system,code,date,lab_result_num_val,lab_result_text_val,units_of_measure,derived_by_TriNetX,source_id
0,fQ,iwZ,LOINC,10501-5,20200304,22.2,NaN,m[IU]/mL,F,EHR
1,fQ,igx,LOINC,10524-7,20130129,NaN,NaN,units,F,EHR
2,fQ,iAN,LOINC,10524-7,20210113,NaN,NaN,units,F,EHR
3,fQ,iwc,LOINC,10834-0,20181119,3.2,NaN,g/L,F,EHR
4,fQ,iAu,LOINC,10834-0,20190122,3.5,NaN,g/L,F,EHR


In [ ]:
# mapping the LOINC codes to their names
term = pd.read_csv(DATA_PATH + "/standardized_terminology.csv", nrows=1000)

term.head(10)

,code_system,code,code_description,path,unit
0,ATC,0,Medications: Anatomical Therapeutic Chemical,0,NaN
1,ATC,A,ALIMENTARY TRACT AND METABOLISM,0/A,NaN
2,ATC,A...,A,0/Other/A...,NaN
3,ATC,A01,STOMATOLOGICAL PREPARATIONS,0/A/A01,NaN
4,ATC,A01A,STOMATOLOGICAL PREPARATIONS,0/A/A01/A01A,NaN
5,ATC,A01AA,Caries prophylactic agents,0/A/A01/A01A/A01AA,NaN
6,ATC,A01AB,Antiinfectives and antiseptics for local oral ...,0/A/A01/A01A/A01AB,NaN
7,ATC,A01AC,Corticosteroids for local oral treatment,0/A/A01/A01A/A01AC,NaN
8,ATC,A01AD,Other agents for local oral treatment,0/A/A01/A01A/A01AD,NaN
9,ATC,A02,DRUGS FOR ACID RELATED DISORDERS,0/A/A02,NaN


In [ ]:
import os

path = DATA_PATH + "/standardized_terminology.csv"

with open(path) as f:
    row_count = sum(1 for line in f) - 1   # minus header

print("Total rows:", row_count)

Total rows: 2221132


In [ ]:
term = pd.read_csv(DATA_PATH + "/standardized_terminology.csv", nrows=2221132)
term["code_system"].value_counts().head(20)

/tmp/ipykernel_3176/2645233508.py:1: DtypeWarning: Columns (1,2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  term = pd.read_csv(DATA_PATH + "/standardized_terminology.csv", nrows=2221132)


,count
code_system,
LOINC,732225
NDC,352413
HGVS,279164
SNOMEDCT_US,256103
RxDrug,137781
ICD-9-CM,106240
ICD-10-CM,100963
ICD-10-PCS,97298
ICD-O,88267


In [ ]:
# extract kidney-related lab tests from the LOINC table
loinc_terms = term[term["code_system"] == "LOINC"]

In [ ]:
# creatinine kidney lab
loinc_terms[loinc_terms["code_description"].str.contains("creatin", case=False, na=False)].head(10)

,code_system,code,code_description,path,unit
699573,LOINC,100152-8,6-oxo-piperidine-2-carboxylate/Creatinine [Mol...,V-LNC/MTHU000998/MTHU000001/MTHU000049/100152-8,umol/mmol{creat}
699574,LOINC,100152-8,6-oxo-piperidine-2-carboxylate/Creatinine [Mol...,V-LNC/MTHU000999/LP29693-6/LP248770-2/100152-8,umol/mmol{creat}
699575,LOINC,100152-8,6-oxo-piperidine-2-carboxylate/Creatinine [Mol...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,umol/mmol{creat}
699576,LOINC,100153-6,6(R+S)-oxo-propylpiperidine-2-carboxylate/Crea...,V-LNC/MTHU000998/MTHU000001/MTHU000049/100153-6,umol/mmol{creat}
699577,LOINC,100153-6,6(R+S)-oxo-propylpiperidine-2-carboxylate/Crea...,V-LNC/MTHU000999/LP29693-6/LP248770-2/100153-6,umol/mmol{creat}
699578,LOINC,100153-6,6(R+S)-oxo-propylpiperidine-2-carboxylate/Crea...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,umol/mmol{creat}
702893,LOINC,101115-4,Leukotriene E4/Creatinine [Mass Ratio] in 24 h...,V-LNC/MTHU000998/MTHU000001/MTHU000049/101115-4,pg/mg{creat}
702894,LOINC,101115-4,Leukotriene E4/Creatinine [Mass Ratio] in 24 h...,V-LNC/MTHU000999/LP29693-6/LP343631-0/LP7784-4...,pg/mg{creat}
702895,LOINC,101115-4,Leukotriene E4/Creatinine [Mass Ratio] in 24 h...,V-LNC/MTHU000999/LP29693-6/LP343631-0/LP7786-9...,pg/mg{creat}
702896,LOINC,101115-4,Leukotriene E4/Creatinine [Mass Ratio] in 24 h...,V-LNC/MTHU000999/LP29693-6/LP7851-1/LP15705-4/...,pg/mg{creat}


In [ ]:
# eGFR
loinc_terms[loinc_terms["code_description"].str.contains("glomerular|egfr", case=False, na=False)].head(10)

,code_system,code,code_description,path,unit
706585,LOINC,102097-3,Glomerular filtration rate/1.73 sq M.predicted...,V-LNC/MTHU000998/MTHU000001/MTHU000049/102097-3,mL/min/{1.73_m2}
706586,LOINC,102097-3,Glomerular filtration rate/1.73 sq M.predicted...,V-LNC/MTHU000999/LP29693-6/LP248770-2/102097-3,mL/min/{1.73_m2}
706587,LOINC,102097-3,Glomerular filtration rate/1.73 sq M.predicted...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,mL/min/{1.73_m2}
712093,LOINC,103542-7,Tubular reabsorption of phosphorus/Glomerular ...,V-LNC/MTHU000998/MTHU000001/MTHU000049/103542-7,mg/dL
712094,LOINC,103542-7,Tubular reabsorption of phosphorus/Glomerular ...,V-LNC/MTHU000999/LP29693-6/LP248770-2/103542-7,mg/dL
712095,LOINC,103542-7,Tubular reabsorption of phosphorus/Glomerular ...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,mg/dL
784129,LOINC,21665-5,EGFR gene mutations found [Identifier] in Bloo...,V-LNC/MTHU000998/MTHU000001/MTHU000100/21665-5,units
784130,LOINC,21665-5,EGFR gene mutations found [Identifier] in Bloo...,V-LNC/MTHU000999/LP29693-6/LP248770-2/21665-5,units
784131,LOINC,21665-5,EGFR gene mutations found [Identifier] in Bloo...,V-LNC/MTHU000999/LP29693-6/LP7822-2/LP32747-5/...,units
784132,LOINC,21665-5,EGFR gene mutations found [Identifier] in Bloo...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,units


In [ ]:
# BUN/UREA
loinc_terms[loinc_terms["code_description"].str.contains("urea|bun", case=False, na=False)].head(10)

,code_system,code,code_description,path,unit
700157,LOINC,100368-0,Amino Acids Urea Cycle Panel - Serum or Plasma...,V-LNC/MTHU000998/MTHU000001/MTHU000114/100368-0,units
700158,LOINC,100368-0,Amino Acids Urea Cycle Panel - Serum or Plasma...,V-LNC/MTHU000999/LP29693-6/LP248770-2/100368-0,units
700159,LOINC,100368-0,Amino Acids Urea Cycle Panel - Serum or Plasma...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,units
701897,LOINC,100830-9,Alpha subunit free [Units/volume] in Serum or ...,V-LNC/MTHU000998/MTHU000001/MTHU000047/100830-9,U/L
701898,LOINC,100830-9,Alpha subunit free [Units/volume] in Serum or ...,V-LNC/MTHU000999/LP29693-6/LP248770-2/100830-9,U/L
701899,LOINC,100830-9,Alpha subunit free [Units/volume] in Serum or ...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,U/L
701900,LOINC,100831-7,Alpha subunit free [Units/volume] in Serum or ...,V-LNC/MTHU000998/MTHU000001/MTHU000047/100831-7,U/L
701901,LOINC,100831-7,Alpha subunit free [Units/volume] in Serum or ...,V-LNC/MTHU000999/LP29693-6/LP248770-2/100831-7,U/L
701902,LOINC,100831-7,Alpha subunit free [Units/volume] in Serum or ...,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP248770...,U/L
701903,LOINC,100832-5,Alpha subunit free [Units/volume] in Serum or ...,V-LNC/MTHU000998/MTHU000001/MTHU000047/100832-5,U/L


In [ ]:
# potassium
loinc_terms[loinc_terms["code_description"].str.contains("potassium", case=False, na=False)].head(10)

,code_system,code,code_description,path,unit
710759,LOINC,10322-6,Potassium intake 24 hour,V-LNC/MTHU000998/MTHU000002/MTHU000092/10322-6,mol/(24.h)
710760,LOINC,10322-6,Potassium intake 24 hour,V-LNC/MTHU000999/LP432695-7/LP7787-7/LP248771-...,mol/(24.h)
710761,LOINC,10322-6,Potassium intake 24 hour,V-LNC/MTHU000999/LP432695-7/LP7787-7/LP7815-6/...,mol/(24.h)
710762,LOINC,10322-6,Potassium intake 24 hour,V-LNC/MTHU000999/LP7787-7/LP248771-0/10322-6,mol/(24.h)
710763,LOINC,10322-6,Potassium intake 24 hour,V-LNC/MTHU000999/LP7787-7/LP7815-6/LP73135-3/L...,mol/(24.h)
718242,LOINC,11148-4,Potassium/Creatinine [Mass Ratio] in Urine,V-LNC/MTHU000998/MTHU000001/MTHU000049/11148-4,units
718243,LOINC,11148-4,Potassium/Creatinine [Mass Ratio] in Urine,V-LNC/MTHU000999/LP29693-6/LP343631-0/LP7786-9...,units
718244,LOINC,11148-4,Potassium/Creatinine [Mass Ratio] in Urine,V-LNC/MTHU000999/LP29693-6/LP343631-0/LP7786-9...,units
718245,LOINC,11148-4,Potassium/Creatinine [Mass Ratio] in Urine,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP343631...,units
718246,LOINC,11148-4,Potassium/Creatinine [Mass Ratio] in Urine,V-LNC/MTHU000999/LP432695-7/LP29693-6/LP343631...,units


# REPLICATING KALYANIS PIPELINE

In [ ]:
# creating a 1 year observation window for procedure table
import pandas as pd
import os

# load procedure file
proc_path = os.path.join(DATA_PATH, "procedure.csv")
procedure = pd.read_csv(proc_path, dtype=str)

# convert dates
procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")
aki_clean["obs_start_date"] = pd.to_datetime(aki_clean["obs_start_date"], errors="coerce")
aki_clean["aki_index_date"] = pd.to_datetime(aki_clean["aki_index_date"], errors="coerce")

# merge with cohort dates
proc_obs = procedure.merge(
    aki_clean[["patient_id", "obs_start_date", "aki_index_date"]],
    on="patient_id",
    how="inner"
)

# keep procedures in 1-year observation window
proc_obs = proc_obs[
    (proc_obs["date"] >= proc_obs["obs_start_date"]) &
    (proc_obs["date"] < proc_obs["aki_index_date"])
]

print(proc_obs.shape)
proc_obs.head()

(2606192, 10)


,patient_id,encounter_id,code_system,code,principal_procedure_indicator,date,derived_by_TriNetX,source_id,obs_start_date,aki_index_date
9,HBD,KBeD,SNOMED,243142003,Unknown,2010-03-13,F,EHR,2009-03-27,2010-03-27
10,HBD,KBeD,SNOMED,243142003,Unknown,2010-03-14,F,EHR,2009-03-27,2010-03-27
11,HBD,KBeD,SNOMED,243142003,Unknown,2010-03-14,F,EHR,2009-03-27,2010-03-27
12,HBD,KBeD,SNOMED,243142003,Unknown,2010-03-18,F,EHR,2009-03-27,2010-03-27
13,HBD,KBeD,SNOMED,243142003,Unknown,2010-03-18,F,EHR,2009-03-27,2010-03-27


In [ ]:
# medication.csv observation window
# load medication file
med_path = os.path.join(DATA_PATH, "medication_drug.csv")
med = pd.read_csv(med_path, dtype=str)

# convert dates
med["start_date"] = pd.to_datetime(med["start_date"], format="%Y%m%d", errors="coerce")

# merge with AKI cohort
med_obs = med.merge(
    aki_clean[["patient_id", "obs_start_date", "aki_index_date"]],
    on="patient_id",
    how="inner"
)

# keep medications in the 1-year observation window
med_obs = med_obs[
    (med_obs["start_date"] >= med_obs["obs_start_date"]) &
    (med_obs["start_date"] < med_obs["aki_index_date"])
]

print(med_obs.shape)
med_obs.head()

(2891995, 15)


,patient_id,encounter_id,unique_id,code_system,code,start_date,route,brand,strength,quantity_dispensed,days_supply,derived_by_TriNetX,source_id,obs_start_date,aki_index_date
28,HBD,KhfD,HRjb,RxNorm,1234995,2010-03-22,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2009-03-27,2010-03-27
42,HhD,KhME,HREi,RxNorm,1654008,2010-05-14,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2010-04-11,2011-04-11
45,HhD,KhME,HhFi,RxNorm,1808224,2010-05-14,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2010-04-11,2011-04-11
48,HhD,KhME,HhIi,RxNorm,1928570,2010-05-12,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2010-04-11,2011-04-11
49,HhD,KhME,HxIi,RxNorm,1928570,2010-05-12,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2010-04-11,2011-04-11


In [ ]:
# diagnosis observation window
diag_obs = diagnosis.merge(
    aki_clean[["patient_id", "obs_start_date", "aki_index_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    (diag_obs["date"] >= diag_obs["obs_start_date"]) &
    (diag_obs["date"] < diag_obs["aki_index_date"])
]

diag_obs.head()

In [ ]:
print(diag_obs.shape)
print(proc_obs.shape)
print(med_obs.shape)

(3287840, 12)
(2606192, 10)
(2891995, 15)


Meaning:

~3.28M diagnosis events

~2.60M procedure events

~2.89M medication events

All within 1 year before AKI.

So the timeline will contain ~8.7M events total

In [ ]:
# converting each table into the same event format so they can be merged

# diagnosis events
diagnosis_events = diag_obs[["patient_id","date","code"]].copy()
diagnosis_events["event_type"] = "diagnosis"
diagnosis_events = diagnosis_events[["patient_id","date","event_type","code"]]

# procedure events
procedure_events = proc_obs[["patient_id","date","code"]].copy()
procedure_events["event_type"] = "procedure"
procedure_events = procedure_events[["patient_id","date","event_type","code"]]

# medication events
medication_events = med_obs[["patient_id","start_date","code"]].copy()
medication_events = medication_events.rename(columns={"start_date":"date"})
medication_events["event_type"] = "medication"
medication_events = medication_events[["patient_id","date","event_type","code"]]

In [ ]:
# merging all events

all_events = pd.concat(
    [diagnosis_events, procedure_events, medication_events],
    ignore_index=True
)

print(all_events.shape)
all_events.head()

(8786027, 4)


,patient_id,date,event_type,code
0,HBD,2010-03-13,diagnosis,R06.9
1,HBD,2010-03-18,diagnosis,R52
2,HBD,2010-03-20,diagnosis,R52
3,HBD,2010-03-12,diagnosis,J18.9
4,HBD,2010-03-13,diagnosis,J18.9


In [ ]:
# sorting events chronologically per patient
all_events = all_events.sort_values(["patient_id", "date"])

all_events.head(20)

,patient_id,date,event_type,code
786507,#A#B,2015-06-17,diagnosis,250.00
786511,#A#B,2015-06-17,diagnosis,305.00
786512,#A#B,2015-06-17,diagnosis,401.0
786514,#A#B,2015-06-17,diagnosis,425.4
786520,#A#B,2015-06-17,diagnosis,496
4218557,#A#B,2015-06-17,procedure,99214
4218565,#A#B,2015-06-17,procedure,G0463
4218566,#A#B,2015-06-17,procedure,G0463
4218567,#A#B,2015-06-17,procedure,G0463
4218568,#A#B,2015-06-17,procedure,G0463


In [ ]:
# grouping the events per patient
patient_sequences = all_events.groupby("patient_id")["code"].apply(list)

print(patient_sequences.shape)
patient_sequences.head()

(27504,)


,code
patient_id,
#A#B,"[250.00, 305.00, 401.0, 425.4, 496, 99214, G04..."
#A#C,"[E11.69, E66.9, H91.92, I10, K21.00, Z76.0, 36..."
#A#D,"[Z79.01, 36416, 85610, 99211, Z79.01, 36416, 8..."
#A0B,"[E11.9, E87.1, F33.2, G56.03, I10, R42, R55, Z..."
#A0C,"[274.00, 274.9, 356.9, 714.9, 735.0, V72.63, 9..."


In [ ]:
# removing sparse patients (very short sequences)
# computing sequence length
seq_lengths = all_events.groupby("patient_id").size()

seq_lengths.head()

,0
patient_id,
#A#B,118
#A#C,336
#A#D,266
#A0B,86
#A0C,76


In [ ]:
seq_lengths.shape

(27504,)

In [ ]:
# keeping patients with more than 3 events
valid_patients = seq_lengths[seq_lengths >= 3].index

In [ ]:
valid_patients.shape

(26813,)

In [ ]:
# filtering the timeline
all_events_filtered = all_events[all_events["patient_id"].isin(valid_patients)]

print(all_events_filtered.shape)

(8785063, 4)


In [ ]:
# converting the codes into sequence tokens for the transformers
# make each event unique by combining type + code
all_events_filtered["event_token"] = (
    all_events_filtered["event_type"].astype(str) + "_" +
    all_events_filtered["code"].astype(str)
)

all_events_filtered.head()

/tmp/ipykernel_3176/3249433694.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_events_filtered["event_token"] = (


,patient_id,date,event_type,code,event_token
786507,#A#B,2015-06-17,diagnosis,250.00,diagnosis_250.00
786511,#A#B,2015-06-17,diagnosis,305.00,diagnosis_305.00
786512,#A#B,2015-06-17,diagnosis,401.0,diagnosis_401.0
786514,#A#B,2015-06-17,diagnosis,425.4,diagnosis_425.4
786520,#A#B,2015-06-17,diagnosis,496,diagnosis_496


In [ ]:
# build vocabulary and numeric sequences
# unique event tokens
unique_tokens = all_events_filtered["event_token"].unique()

# token -> integer mapping
token2id = {token: i + 1 for i, token in enumerate(unique_tokens)}   # start from 1
id2token = {i: token for token, i in token2id.items()}

print("Vocabulary size:", len(token2id))

Vocabulary size: 60254


In [ ]:
# converting each patient timeline into integer sequences
patient_sequences = (
    all_events_filtered
    .groupby("patient_id")["event_token"]
    .apply(list)
)

patient_sequences_ids = patient_sequences.apply(
    lambda seq: [token2id[token] for token in seq]
)

print(patient_sequences_ids.shape)
patient_sequences_ids.head()

(26813,)


,event_token
patient_id,
#A#B,"[1, 2, 3, 4, 5, 6, 7, 7, 7, 7, 7, 7, 7, 7, 8, ..."
#A#C,"[47, 48, 49, 50, 51, 52, 53, 54, 55, 6, 56, 57..."
#A#D,"[185, 54, 186, 187, 185, 54, 186, 187, 188, 10..."
#A0B,"[110, 111, 261, 262, 50, 20, 263, 264, 53, 53,..."
#A0C,"[291, 292, 293, 294, 295, 296, 297, 297, 298, ..."


In [ ]:
# making sure all_events_filtered was sorted by patient id and date before grouping
all_events = all_events.sort_values(["patient_id", "date"])

In [ ]:
# attaching the label Y to each patient sequence
# create patient-level label table
labels = aki_clean[["patient_id", "Y_esrd_or_dialysis_2yr"]].drop_duplicates()

# turn sequence series into dataframe
seq_df = patient_sequences_ids.reset_index()
seq_df.columns = ["patient_id", "sequence"]

# merge sequences with labels
seq_df = seq_df.merge(labels, on="patient_id", how="inner")

print(seq_df.shape)
seq_df.head()

(26813, 3)


,patient_id,sequence,Y_esrd_or_dialysis_2yr
0,#A#B,"[1, 2, 3, 4, 5, 6, 7, 7, 7, 7, 7, 7, 7, 7, 8, ...",0
1,#A#C,"[47, 48, 49, 50, 51, 52, 53, 54, 55, 6, 56, 57...",1
2,#A#D,"[185, 54, 186, 187, 185, 54, 186, 187, 188, 10...",0
3,#A0B,"[110, 111, 261, 262, 50, 20, 263, 264, 53, 53,...",0
4,#A0C,"[291, 292, 293, 294, 295, 296, 297, 297, 298, ...",0


In [ ]:
# train test split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    seq_df,
    test_size=0.2,
    random_state=42,
    stratify=seq_df["Y_esrd_or_dialysis_2yr"]
)

print(train_df.shape, test_df.shape)

(21450, 3) (5363, 3)


In [ ]:
# finding the sequence lenghts before padding
train_df["seq_len"] = train_df["sequence"].apply(len)
test_df["seq_len"] = test_df["sequence"].apply(len)

print(train_df["seq_len"].describe())
print(test_df["seq_len"].describe())

count    21450.000000
mean       328.032681
std        752.821206
min          3.000000
25%         62.250000
50%        161.000000
75%        372.000000
max      53006.000000
Name: seq_len, dtype: float64
count     5363.000000
mean       326.079060
std        570.645798
min          3.000000
25%         64.000000
50%        164.000000
75%        379.500000
max      14871.000000
Name: seq_len, dtype: float64


In [ ]:
# padding truncating sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_SEQ_LEN = 400

X_train = pad_sequences(
    train_df["sequence"],
    maxlen=MAX_SEQ_LEN,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    test_df["sequence"],
    maxlen=MAX_SEQ_LEN,
    padding="post",
    truncating="post"
)

y_train = train_df["Y_esrd_or_dialysis_2yr"].values
y_test = test_df["Y_esrd_or_dialysis_2yr"].values

print(X_train.shape)
print(X_test.shape)

(21450, 400)
(5363, 400)


# Building Transformers

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

VOCAB_SIZE = len(token2id) + 1   # +1 because padding token = 0
MAX_SEQ_LEN = 400
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

def build_transformer_model(maxlen, vocab_size, embed_dim=128, num_heads=4, ff_dim=128, rate=0.1):
    inputs = layers.Input(shape=(maxlen,))
    embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
    x = embedding_layer(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim, rate)
    x = transformer_block(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(rate)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(rate)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_transformer_model(
    maxlen=MAX_SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    rate=DROPOUT
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 400, 128)       │     7,763,840 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 400, 128)       │       297,344 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,069,505 (30.78 MB)

 Trainable params: 8,069,505 (30.78 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf

def binary_focal_loss(alpha=0.75, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

        loss_val = -alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)
        return tf.reduce_mean(loss_val)

    return loss

In [ ]:
# compiling focal loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=binary_focal_loss(alpha=0.75, gamma=2.0),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=3,
    mode="max",
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 89s 95ms/step - accuracy: 0.8803 - auc: 0.5253 - loss: 0.0461 - val_accuracy: 0.9156 - val_auc: 0.6286 - val_loss: 0.0396
Epoch 2/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8989 - auc: 0.6992 - loss: 0.0373 - val_accuracy: 0.9042 - val_auc: 0.6710 - val_loss: 0.0376
Epoch 3/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9081 - auc: 0.8871 - loss: 0.0257 - val_accuracy: 0.8308 - val_auc: 0.6457 - val_loss: 0.0445
Epoch 4/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9323 - auc: 0.9579 - loss: 0.0169 - val_accuracy: 0.8385 - val_auc: 0.6406 - val_loss: 0.0666
Epoch 5/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9548 - auc: 0.9856 - loss: 0.0100 - val_accuracy: 0.8438 - val_auc: 0.6213 - val_loss: 0.1009


In [ ]:
# evaluate test set
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

168/168 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step
Accuracy: 0.9028528808502704
ROC AUC: 0.6680648689779807
F1 Score: 0.13884297520661157
              precision    recall  f1-score   support

           0       0.92      0.98      0.95      4890
           1       0.32      0.09      0.14       473

    accuracy                           0.90      5363
   macro avg       0.62      0.54      0.54      5363
weighted avg       0.86      0.90      0.88      5363

